Το συγκεκριμενο Notebook τρεχει μονο οταν ερχονται historical payment για να τα μεταφερει στον reference table 

In [0]:
env = "p"

In [0]:
from datetime import datetime
from pyspark.sql.functions import lit, concat, col, coalesce, to_date, year, month, split,regexp_replace
from pyspark.sql.types import *

def getDateTime():
    return datetime.now().strftime("%Y-%m-%d")

In [0]:
dbutils.widgets.text('IngestionDate', 'none', '01-Ingestion Date')
dbutils.widgets.text('pathfile', 'none', '02-Path File')
dbutils.widgets.text('portfolioname', 'none', '03-Portfolio Name')

IngestionDate = dbutils.widgets.get('IngestionDate')
pathfile = dbutils.widgets.get('pathfile')
portfolioname = dbutils.widgets.get('portfolioname')

In [0]:
#get specific column from csv

df_raw = spark.read \
    .option("header", "true") \
    .option("sep", ";") \
    .csv(pathfile)

df_initial = df_raw.select(
    "Loan ID (current)",
    "Financial Event Type Description",
    "Transaction Date (MM/YY)",
    "Transaction Amount"
)


In [0]:
#rename  stilwn kai prosthesi allwn
df_rename = (
    df_initial
    .withColumnRenamed("Loan ID (current)", "Account")
    .withColumnRenamed("Financial Event Type Description", "PaymentType")
    .withColumnRenamed("Payment_date", "PaymentDate")
    .withColumnRenamed("Transaction Amount", "Amount")
    .withColumnRenamed("Prefix", "PortfolioPrefix")
    .withColumn("IngestionDate", lit(IngestionDate)) 
    .withColumn("portfolioName", lit(portfolioname))
    .withColumn("Principal", lit(None))
    .withColumn("Interest", lit(None))
    .withColumn("Penalty", lit(None))

)

In [0]:
# from pyspark.sql.functions import col, lit, regexp_replace
from pyspark.sql.types import *

def align_to_delta_table(df, table_name):
    
    target_df = spark.table(table_name)
    target_schema = target_df.schema
    
    df_cols_lower = {c.lower(): c for c in df.columns}
    
    aligned_df = df
    
    for field in target_schema:
        target_col = field.name
        target_type = field.dataType
        
        if target_col.lower() in df_cols_lower:
            source_col = df_cols_lower[target_col.lower()]
            
            # Rename if needed
            if source_col != target_col:
                aligned_df = aligned_df.withColumnRenamed(source_col, target_col)
            
            source_type = dict(df.dtypes)[source_col]
            
            # ---- Numeric handling for European format ----
            if isinstance(target_type, (DoubleType, FloatType, DecimalType, IntegerType, LongType)):
                
                if source_type == "string":
                    
                    # Remove thousands separator "." and convert decimal "," to "."
                    aligned_df = aligned_df.withColumn(
                        target_col,
                        regexp_replace(
                            regexp_replace(col(target_col), r"\.", ""),
                            ",",
                            "."
                        ).cast(target_type)
                    )
                else:
                    # Already numeric → simple cast
                    aligned_df = aligned_df.withColumn(
                        target_col,
                        col(target_col).cast(target_type)
                    )
            
            else:
                # Non-numeric columns
                aligned_df = aligned_df.withColumn(
                    target_col,
                    col(target_col).cast(target_type)
                )
        
        else:
            # Column missing → add null
            aligned_df = aligned_df.withColumn(
                target_col,
                lit(None).cast(target_type)
            )
    
    # Reorder columns exactly as target table
    aligned_df = aligned_df.select([f.name for f in target_schema])
    
    return aligned_df


df_final = align_to_delta_table(df_rename, table_name)

In [0]:
table_name = "`{env}-cdh`.reference.premigrationpayments"

# pairnoume ola ta distinct PortfolioName
portfolio_values = [
    row["PortfolioName"]
    for row in df_final.select("PortfolioName").distinct().collect()
]

if portfolio_values:
    portfolios_sql = ",".join([f"'{p}'" for p in portfolio_values])

    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", f"PortfolioName IN ({portfolios_sql})") \
        .saveAsTable(table_name)